# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis

One row represents the daily performance of one pseudonymized content item for one pseudonymized client on a single report date.

### Time Window

This notebook uses the March 2026 partition (`month = '2026-03'`) for verification. A mid-panel month is used instead of the final month, following the assignment guidance to avoid developing on the natural outcome window.

### Verification Summary

The verification query confirms that the selected partition contains **9,841,378 rows** covering the period **2026-03-01** through **2026-03-31**.

In [9]:
from google.colab import userdata
import duckdb

# Connect to DuckDB
con = duckdb.connect()

# Authenticate with Hugging Face (HF_TOKEN must be stored in Colab Secrets)
con.execute(f"""
CREATE OR REPLACE SECRET (
    TYPE huggingface,
    TOKEN '{userdata.get("flyrank")}'
)
""")

# Dataset location
rel = "hf://datasets/FlyRank/internship-warehouse"

# Verify the analysis window (March 2026)
summary = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

print(summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows first_report_date last_report_date
0     9841378        2026-03-01       2026-03-31


### Listing Available Partitions in Hugging Face Dataset

To find the correct `month=` partitions, we can use the `huggingface_hub` library to inspect the dataset repository directly. First, we need to install it.

## 2. Fields: feature / label / context / excluded

### Features

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_sessions`
- `ga4_users`

These variables are available on or before the report date and can be used as predictive features.

### Label / Proxy

Future search performance (to be defined during the modeling phase). No future outcome variables are used as features in this notebook.

### Context

- `client_hash_id`
- `content_hash_id`
- `report_date`
- `month`

These columns identify records or define the analysis window. They are used for grouping, filtering, and interpretation rather than prediction.

### Excluded

- Future performance metrics, because they would introduce target leakage.
- Hash identifiers as predictive features, because they are identifiers rather than meaningful signals.
- Rows where `ga4_data_available IS FALSE` when creating GA4-based features, because zero-filled values before GA4 availability do not represent real user activity.

In [10]:
fields = {
    "Feature": [
        "gsc_clicks",
        "gsc_impressions",
        "gsc_ctr",
        "gsc_avg_position",
        "ga4_sessions"
    ],

    "Label / Proxy": [
        "Future search performance (defined later during modeling)"
    ],

    "Context": [
        "client_hash_id",
        "content_hash_id",
        "report_date"
    ],

    "Excluded": [
        "Future metrics (would leak future information)",
        "Identifiers (used only for grouping and joins)"
    ]
}

fields

{'Feature': ['gsc_clicks',
  'gsc_impressions',
  'gsc_ctr',
  'gsc_avg_position',
  'ga4_sessions'],
 'Label / Proxy': ['Future search performance (defined later during modeling)'],
 'Context': ['client_hash_id', 'content_hash_id', 'report_date'],
 'Excluded': ['Future metrics (would leak future information)',
  'Identifiers (used only for grouping and joins)']}

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
# Query 1 — Verify the grain
grain = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print("Query 1: Grain Verification")
display(grain)

if grain.empty:
    print("✓ No duplicate report_date-client-content combinations found. The documented grain is verified.")


# Query 2 — Verify row count and date window
counts = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

print("\nQuery 2: Row Count and Date Window")
display(counts)


# Query 3 — Verify data availability using IS TRUE
availability = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

print("\nQuery 3: Data Availability")
display(availability)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1: Grain Verification


,report_date,client_hash_id,content_hash_id,duplicate_rows


✓ No duplicate report_date-client-content combinations found. The documented grain is verified.

Query 2: Row Count and Date Window


,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Query 3: Data Availability


,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


## 4. Data limits

This warehouse records observed search and analytics performance but cannot explain why rankings or engagement changed.

Client histories begin on different dates, resulting in an unbalanced panel where some clients have substantially longer histories than others.

Rows before GA4 collection are zero-filled and must be interpreted using the `ga4_data_available` flag rather than assuming zero engagement.

These observations support descriptive and predictive analysis but should not be interpreted as causal evidence.

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.